<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** n/a (worked attempt, not timed)  **Stopped at:** n/a  **Furthest step I reached (see the brief):** step 4 (gold table built and checked against the source). Step 5 (stretch) not attempted.

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

**What I am doing here:** for each table I check (a) what one row is, (b) whether the key is really unique, (c) whether every key finds a match in the table it joins to, and (d) the odd values in the columns the brief needs. Nothing is changed in this section; I only look.

In [1]:
# Short names for the six tables
orders = tables["orders"]
events = tables["order_status_events"]
lines = tables["order_lines"]
payments = tables["payments"]
restaurants = tables["restaurants"]
customers = tables["customers"]

for name, df in tables.items():
    print(f"{name:20s} {df.shape[0]:>6,} rows | missing values: {df.isna().sum().sum()} | exact duplicate rows: {df.duplicated().sum()}")

orders               14,494 rows | missing values: 0 | exact duplicate rows: 0
order_status_events  56,812 rows | missing values: 0 | exact duplicate rows: 0
order_lines          36,120 rows | missing values: 0 | exact duplicate rows: 0
payments             16,423 rows | missing values: 0 | exact duplicate rows: 30
restaurants              67 rows | missing values: 0 | exact duplicate rows: 0
customers             4,200 rows | missing values: 0 | exact duplicate rows: 0


### 1a. Keys: is each table's key unique?

In [2]:
print("orders.order_id unique:          ", orders["order_id"].is_unique)
print("events.event_id unique:          ", events["event_id"].is_unique)
print("payments.payment_id unique:      ", payments["payment_id"].is_unique)
print("order_lines (order_id, line_no): ", not lines.duplicated(["order_id", "line_no"]).any())
print("restaurants.restaurant_id unique:", restaurants["restaurant_id"].is_unique)
print("customers.customer_id unique:    ", customers["customer_id"].is_unique)

orders.order_id unique:           True
events.event_id unique:           True
payments.payment_id unique:       False
order_lines (order_id, line_no):  True
restaurants.restaurant_id unique: False
customers.customer_id unique:     True


Two keys are **not** unique: `payment_id` and `restaurant_id`. I look at each.

In [3]:
# Payments: are the repeated payment_ids whole-row copies?
dup_pay = payments[payments["payment_id"].duplicated(keep=False)].sort_values("payment_id")
print(len(dup_pay), "rows share a payment_id;", payments.duplicated().sum(), "are exact copies of another row")
dup_pay.head(4)

60 rows share a payment_id; 30 are exact copies of another row


      payment_id order_id payment_type  amount              paid_at
5949     P000219  O000185       charge   25.49  2026-03-02 19:05:43
13373    P000219  O000185       charge   25.49  2026-03-02 19:05:43
1673     P000927  O000815       charge   23.99  2026-03-08 11:53:53
4665     P000927  O000815       charge   23.99  2026-03-08 11:53:53

In [4]:
# Restaurants: 67 rows but only 64 restaurants. Which appear twice?
restaurants[restaurants["restaurant_id"].duplicated(keep=False)].sort_values(["restaurant_id", "effective_from"])

   restaurant_id   restaurant_name   cuisine        city effective_from
3           R007  Royal Japanese 7  Japanese     Bristol     2025-01-01
7           R007  Royal Japanese 7  Japanese      London     2026-04-14
51          R021     Grand Thai 21      Thai  Manchester     2025-01-01
6           R021     Grand Thai 21      Thai       LEEDS     2026-05-06
48          R040   Corner Vegan 40     Vegan      London     2025-01-01
35          R040   Corner Vegan 40     Vegan     Bristol     2026-05-27

Three restaurants (R007, R021, R040) **moved city** during the period. `effective_from` says when each version starts. So an order must take the restaurant's city **as it was on the order date**, not the latest one. A plain join on `restaurant_id` would duplicate those restaurants' orders.

### 1b. Joins: does every key find a match?

In [5]:
print("orders with unknown restaurant:", (~orders["restaurant_id"].isin(restaurants["restaurant_id"])).sum())
print("orders with unknown customer:  ", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("events not matching an order:  ", (~events["order_id"].isin(orders["order_id"])).sum())
print("lines not matching an order:   ", (~lines["order_id"].isin(orders["order_id"])).sum())
print("payments not matching an order:", (~payments["order_id"].isin(orders["order_id"])).sum())
print("orders with no lines:          ", (~orders["order_id"].isin(lines["order_id"])).sum())
print("orders with no payment:        ", (~orders["order_id"].isin(payments["order_id"])).sum())

orders with unknown restaurant: 0
orders with unknown customer:   0
events not matching an order:   0
lines not matching an order:    0
payments not matching an order: 0
orders with no lines:           0
orders with no payment:         0


### 1c. Categories: cuisine and city labels

In [6]:
print(restaurants["cuisine"].value_counts().to_dict())
print(restaurants["city"].value_counts().to_dict())

{'Vegan': 9, 'Thai': 8, 'Chinese': 8, 'Burgers': 7, 'Japanese': 7, 'Indian': 7, 'Italian': 7, 'Mexican': 6, 'mexican': 2, 'japanese': 1, 'thai': 1, 'Burgers ': 1, 'Japanese ': 1, 'Indian ': 1, 'Italian ': 1}
{'London': 23, 'Manchester': 17, 'Birmingham': 12, 'Bristol': 7, 'Leeds': 5, 'LEEDS': 1, 'MANCHESTER': 1, 'LONDON': 1}


### 1d. Status events

In [7]:
events["status"].value_counts()

status
placed       14494
accepted     14154
picked_up    13745
delivered    13549
cancelled      870

In [8]:
# The same status recorded twice for one order?
dup_events = events[events.duplicated(["order_id", "status"], keep=False)].sort_values(["order_id", "status", "event_ts"])
print(events.duplicated(["order_id", "status"]).sum(), "repeated (order, status) events, by status:")
print(events[events.duplicated(["order_id", "status"])]["status"].value_counts().to_dict())
dup_events.head(6)

360 repeated (order, status) events, by status:
{'delivered': 140, 'picked_up': 121, 'accepted': 99}


      event_id order_id     status              event_ts
33354  E000037  O000011   accepted  2026-03-01T12:15:11Z
42128  E000038  O000011   accepted  2026-03-01T12:15:11Z
30265  E000088  O000019  delivered  2026-03-01T13:09:30Z
30881  E000091  O000019  delivered  2026-03-01T13:11:18Z
251    E000080  O000021  picked_up  2026-03-01T13:04:32Z
25724  E000079  O000021  picked_up  2026-03-01T13:04:32Z

### 1e. Timestamps
The brief says every timestamp is UTC. `order_ts` and `event_ts` end in `Z` (UTC). `paid_at` has no time-zone marker, so I check it lines up with the order time.

In [9]:
order_ts = pd.to_datetime(orders.set_index("order_id")["order_ts"], utc=True)
charges = payments[payments["payment_type"] == "charge"].drop_duplicates().set_index("order_id")
charge_ts = pd.to_datetime(charges["paid_at"]).dt.tz_localize("UTC")

lag_seconds = (charge_ts - order_ts).dt.total_seconds()
print("seconds from order to charge:", lag_seconds.describe()[["min", "50%", "max"]].to_dict())
print("orders run from", order_ts.min(), "to", order_ts.max())

seconds from order to charge: {'min': 20.0, '50%': 55.0, 'max': 89.0}
orders run from 2026-03-01 09:13:23+00:00 to 2026-06-30 22:58:46+00:00


### 1f. Money: does what was charged match what was ordered?

In [10]:
lines["line_value"] = lines["quantity"] * lines["unit_price"]
food = lines.groupby("order_id")["line_value"].sum()
expected = food + orders.set_index("order_id")["delivery_fee"]

print("charges per order:", charges.groupby("order_id").size().value_counts().to_dict())
print("orders where charge = food + delivery fee:", ((charges["amount"] - expected).round(2) == 0).sum(), "of", len(orders))
payments.groupby("payment_type")["amount"].describe()[["count", "min", "max"]]

charges per order: {1: 14494}
orders where charge = food + delivery fee: 14494 of 14494


                count     min     max
payment_type                         
charge        14524.0    2.00  127.49
refund         1899.0 -127.49   -0.35

**What I found**

| Table | One row is | Odd or missing |
| --- | --- | --- |
| `orders` (14,494) | one order | Clean: unique `order_id`, no missing values. Orders run from 1 March to 30 June 2026 (UTC). |
| `order_status_events` (56,812) | one status change for one order | 360 repeated (order, status) events, some with timestamps up to 3 minutes apart. 870 orders are `cancelled`. |
| `order_lines` (36,120) | one item line in an order | Clean. Each item has a single price. |
| `payments` (16,423) | one money movement (charge or refund) | 30 exact duplicate rows (a re-sent file). Refunds are stored as negative amounts. |
| `restaurants` (67) | one **version** of a restaurant | 3 restaurants moved city, so 64 restaurants have 67 rows. Cuisine and city have inconsistent case and trailing spaces (`mexican`, `Burgers `, `LEEDS`). |
| `customers` (4,200) | one customer | Clean. Not needed for this brief. |

**Good news:**
- Every key matches.
- Every order has exactly one charge (after de-duplication).
- Every charge equals food plus delivery fee to the penny.
- `paid_at` is 20 to 89 seconds after `order_ts`, so it is UTC too, as the brief says.

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

**One row of the final (gold) table** = one week × one cuisine × one city. It holds the number of orders placed that week at restaurants of that cuisine in that city, their net revenue, and their average delivery time.

**Decisions**

| Question in the brief | My decision | Why |
| --- | --- | --- |
| Which week does an order belong to? | The Monday-to-Sunday week of the order time **in UK local time** (`Europe/London`) | Plateful runs in the UK and its report will be read in UK time. Timestamps are UTC, and the clocks went forward on 29 March, so a late-evening Sunday order could otherwise land in the wrong week (I count how many in section 3). |
| Which orders count? | Every order that was **not cancelled** | A cancelled order was fully refunded, so it is not trade. Orders that were picked up but never marked delivered are kept: the customer paid and was not refunded. |
| What is net revenue? | Charges minus refunds for the order (food + delivery fee − refunds), assigned to the **order's** week | This is the money Plateful kept from customers. We have no commission data, so this is gross customer revenue, not Plateful's profit. Refunds arrive 0 to 2 days later; assigning them to the order's week keeps each order's figures together. |
| What is delivery time? | Minutes from `placed` to the **first** `delivered` event | It is what the customer experiences. Only measured where the time is believable (above 0 and up to 180 minutes); other orders still count, just not in the average. |

**Build order and expected rows**

1. `silver_restaurants`: cleaned labels; still 67 rows (versions).
2. `silver_payments`: de-duplicated, one row per order with `charge`, `refund`, `net_revenue`. **14,494 rows**.
3. `silver_status`: de-duplicated events, one row per order with the first time of each status. **14,494 rows**.
4. `silver_orders`: orders + status + payments + the restaurant version in force on the order date. **14,494 rows**: every join must keep this count.
5. `gold_weekly_ops`: filter out cancelled orders, group by week, cuisine and city.

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Each step reads from the raw tables (never changes them) and prints a row count, so I can see it matches the plan.

### Step 1: `silver_restaurants`
Clean the labels: strip spaces and use title case (`LEEDS` → `Leeds`, `Burgers ` → `Burgers`). Keep every version.

In [11]:
silver_restaurants = restaurants.copy()
silver_restaurants["cuisine"] = silver_restaurants["cuisine"].str.strip().str.title()
silver_restaurants["city"] = silver_restaurants["city"].str.strip().str.title()
silver_restaurants["effective_from"] = pd.to_datetime(silver_restaurants["effective_from"])

print(len(silver_restaurants), "rows")
print(sorted(silver_restaurants["cuisine"].unique()))
print(sorted(silver_restaurants["city"].unique()))

67 rows
['Burgers', 'Chinese', 'Indian', 'Italian', 'Japanese', 'Mexican', 'Thai', 'Vegan']
['Birmingham', 'Bristol', 'Leeds', 'London', 'Manchester']


8 cuisines and 5 cities, as the brief says.

### Step 2: `silver_payments` (one row per order)
Drop the 30 exact duplicates, then total charges and refunds for each order.

In [12]:
pay = payments.drop_duplicates()
print(len(payments), "->", len(pay), "payment rows")

silver_payments = pay.pivot_table(index="order_id", columns="payment_type", values="amount",
                                  aggfunc="sum", fill_value=0).reset_index()
silver_payments.columns.name = None
silver_payments["net_revenue"] = silver_payments["charge"] + silver_payments["refund"]   # refunds are negative

print(len(silver_payments), "orders")
silver_payments.head()

16423 -> 16393 payment rows
14494 orders


  order_id  charge  refund  net_revenue
0  O000001   37.49     0.0        37.49
1  O000002   31.99     0.0        31.99
2  O000003   12.50     0.0        12.50
3  O000004   46.99     0.0        46.99
4  O000005    6.00    -6.0         0.00

### Step 3: `silver_status` (one row per order)
Turn the events into one row per order, with one column per status. Where a status was recorded twice I keep the **first** time (`min`), which is when the status really happened.

In [13]:
ev = events.copy()
ev["event_ts"] = pd.to_datetime(ev["event_ts"], utc=True)

silver_status = ev.pivot_table(index="order_id", columns="status", values="event_ts", aggfunc="min").reset_index()
silver_status.columns.name = None
silver_status = silver_status.rename(columns={"placed": "placed_ts", "accepted": "accepted_ts",
                                              "picked_up": "picked_up_ts", "delivered": "delivered_ts",
                                              "cancelled": "cancelled_ts"})
print(len(silver_status), "orders")

14494 orders


In [14]:
# Final status of each order
silver_status["is_cancelled"] = silver_status["cancelled_ts"].notna()
silver_status["is_delivered"] = silver_status["delivered_ts"].notna()

print("cancelled:", silver_status["is_cancelled"].sum())
print("delivered:", silver_status["is_delivered"].sum())
print("neither:  ", (~silver_status["is_cancelled"] & ~silver_status["is_delivered"]).sum())
print("both:     ", (silver_status["is_cancelled"] & silver_status["is_delivered"]).sum())

cancelled: 870
delivered: 13409
neither:   215
both:      0


No order is both cancelled and delivered. **215 orders are neither**: each was picked up and then nothing more was recorded. They are spread across all four months (so they are not just recent orders still in progress) and none was refunded, so I keep them as orders but they have no delivery time.

In [15]:
# Delivery time in minutes: placed -> first delivered
silver_status["delivery_min"] = (silver_status["delivered_ts"] - silver_status["placed_ts"]).dt.total_seconds() / 60

bins = [-1000, 0, 20, 60, 90, 180, 2000]
silver_status["delivery_min"].pipe(pd.cut, bins).value_counts().sort_index()

delivery_min
(-1000, 0]        16
(0, 20]           86
(20, 60]       13241
(60, 90]          54
(90, 180]          0
(180, 2000]       12

In [16]:
# The negative ones: delivered BEFORE the order was placed
silver_status.loc[silver_status["delivery_min"] < 0,
                  ["order_id", "placed_ts", "picked_up_ts", "delivered_ts", "delivery_min"]].head()

     order_id                 placed_ts              picked_up_ts              delivered_ts  delivery_min
751   O000752 2026-03-07 18:54:09+00:00 2026-03-07 19:08:33+00:00 2026-03-07 18:32:09+00:00         -22.0
1493  O001494 2026-03-14 00:13:31+00:00 2026-03-14 00:35:33+00:00 2026-03-13 23:33:31+00:00         -40.0
2154  O002155 2026-03-19 18:21:59+00:00 2026-03-19 18:35:09+00:00 2026-03-19 17:56:59+00:00         -25.0
2435  O002436 2026-03-21 19:10:00+00:00 2026-03-21 19:25:20+00:00 2026-03-21 18:15:00+00:00         -55.0
3928  O003929 2026-04-03 11:27:35+00:00 2026-04-03 11:49:16+00:00 2026-04-03 10:50:35+00:00         -37.0

In [17]:
# Flag which delivery times are believable: more than 0 and no more than 3 hours
silver_status["delivery_time_valid"] = silver_status["delivery_min"].between(0, 180, inclusive="right")

print("valid delivery times:  ", silver_status["delivery_time_valid"].sum())
print("delivered but excluded:", (silver_status["is_delivered"] & ~silver_status["delivery_time_valid"]).sum(),
      "(16 negative + 12 over 3 hours)")

valid delivery times:   13381
delivered but excluded: 28 (16 negative + 12 over 3 hours)


The 16 negative times have a `delivered` event before `placed` and before `picked_up`, so that timestamp is wrong. The 12 over 3 hours run up to 20 hours, which is more likely a courier forgetting to mark the order delivered than a real wait. I keep all 28 orders (they still count as orders and revenue) but leave them out of the average.

### Step 4: `silver_orders` (one row per order)
Add the UK-local week, then join status and payments. Every join must keep 14,494 rows.

In [18]:
silver_orders = orders.copy()
silver_orders["order_ts"] = pd.to_datetime(silver_orders["order_ts"], utc=True)
silver_orders["order_ts_uk"] = silver_orders["order_ts"].dt.tz_convert("Europe/London")
silver_orders["order_date_uk"] = silver_orders["order_ts_uk"].dt.tz_localize(None).dt.normalize()

# Week = the Monday that starts the order's week, in UK time
silver_orders["week_start"] = silver_orders["order_date_uk"] - pd.to_timedelta(silver_orders["order_date_uk"].dt.dayofweek, unit="D")

In [19]:
# How many orders would land in a different week if I had used UTC?
utc_date = silver_orders["order_ts"].dt.tz_localize(None).dt.normalize()
utc_week = utc_date - pd.to_timedelta(utc_date.dt.dayofweek, unit="D")
print((utc_week != silver_orders["week_start"]).sum(), "orders change week because of the UK/UTC difference")

67 orders change week because of the UK/UTC difference


In [20]:
silver_orders = silver_orders.merge(silver_status, on="order_id", how="left")
print("after joining status:  ", len(silver_orders))

silver_orders = silver_orders.merge(silver_payments, on="order_id", how="left")
print("after joining payments:", len(silver_orders))

after joining status:   14494
after joining payments: 14494


**Restaurant join, as of the order date.** For each order I take the restaurant version with the latest `effective_from` that is on or before the order date. `merge_asof` does exactly this (both tables must be sorted by date).

In [21]:
silver_orders = pd.merge_asof(
    silver_orders.sort_values("order_date_uk"),
    silver_restaurants.sort_values("effective_from")[["restaurant_id", "effective_from", "cuisine", "city"]],
    left_on="order_date_uk",
    right_on="effective_from",
    by="restaurant_id",
    direction="backward",
)
print("after joining restaurants:", len(silver_orders))
print("orders with no cuisine/city:", silver_orders["city"].isna().sum())

after joining restaurants: 14494
orders with no cuisine/city: 0


In [22]:
# Did the moved restaurants get the right city on each side of the move?
moved = silver_orders[silver_orders["restaurant_id"].isin(["R007", "R021", "R040"])]
moved.groupby(["restaurant_id", "effective_from", "city"]).agg(
    first_order=("order_date_uk", "min"), last_order=("order_date_uk", "max"), orders=("order_id", "count"))

                                        first_order last_order  orders
restaurant_id effective_from city                                     
R007          2025-01-01     Bristol     2026-03-05 2026-04-13      23
              2026-04-14     London      2026-04-14 2026-06-29      31
R021          2025-01-01     Manchester  2026-03-01 2026-05-05      97
              2026-05-06     Leeds       2026-05-06 2026-06-30      69
R040          2025-01-01     London      2026-03-05 2026-05-26      97
              2026-05-27     Bristol     2026-05-27 2026-06-30      41

In [23]:
# Keep the columns the report needs, one row per order
silver_orders = silver_orders[[
    "order_id", "customer_id", "restaurant_id", "cuisine", "city",
    "order_ts", "order_ts_uk", "week_start",
    "is_cancelled", "is_delivered", "delivery_min", "delivery_time_valid",
    "delivery_fee", "charge", "refund", "net_revenue",
]].sort_values("order_id").reset_index(drop=True)

print(len(silver_orders), "rows | order_id unique:", silver_orders["order_id"].is_unique)
silver_orders.head()

14494 rows | order_id unique: True


  order_id customer_id restaurant_id  cuisine        city                  order_ts               order_ts_uk week_start  is_cancelled  is_delivered  delivery_min  delivery_time_valid  delivery_fee  charge  refund  net_revenue
0  O000001      C01161          R052  Burgers      London 2026-03-01 09:13:23+00:00 2026-03-01 09:13:23+00:00 2026-02-23         False          True     60.266667                 True          1.49   37.49     0.0        37.49
1  O000002      C00723          R027  Chinese      London 2026-03-01 09:15:33+00:00 2026-03-01 09:15:33+00:00 2026-02-23         False          True     46.333333                 True          2.49   31.99     0.0        31.99
2  O000003      C00781          R044  Burgers  Birmingham 2026-03-01 09:23:42+00:00 2026-03-01 09:23:42+00:00 2026-02-23         False          True     39.050000                 True          0.00   12.50     0.0        12.50
3  O000004      C01430          R049  Italian  Birmingham 2026-03-01 09:28:14+00:00 2026-03-

### Step 5: `gold_weekly_ops`
Keep non-cancelled orders, then group by week, cuisine and city. I also show how many orders the average delivery time is based on (`timed_orders`), so a reader can see when an average rests on only a few orders.

In [24]:
counted = silver_orders[~silver_orders["is_cancelled"]].copy()
counted["valid_delivery_min"] = counted["delivery_min"].where(counted["delivery_time_valid"])   # blank if not believable

gold_weekly_ops = (
    counted
    .groupby(["week_start", "cuisine", "city"])
    .agg(
        orders=("order_id", "count"),
        net_revenue=("net_revenue", "sum"),
        avg_delivery_min=("valid_delivery_min", "mean"),
        timed_orders=("valid_delivery_min", "count"),
    )
    .round({"net_revenue": 2, "avg_delivery_min": 1})
    .reset_index()
)

print(len(gold_weekly_ops), "rows")
gold_weekly_ops.head(10)

555 rows


  week_start  cuisine        city  orders  net_revenue  avg_delivery_min  timed_orders
0 2026-02-23  Burgers  Birmingham       4       125.99              38.8             4
1 2026-02-23  Burgers     Bristol       3        66.77              40.2             3
2 2026-02-23  Burgers      London      11       396.42              39.7            11
3 2026-02-23  Chinese  Birmingham       6       117.46              34.0             6
4 2026-02-23  Chinese      London       7       284.93              35.1             7
5 2026-02-23  Chinese  Manchester       1        24.99              33.7             1
6 2026-02-23   Indian  Birmingham       1        43.99              42.6             1
7 2026-02-23   Indian      London       7       215.56              30.7             7
8 2026-02-23  Italian  Birmingham       7       138.94              36.2             7
9 2026-02-23  Italian       Leeds       1        49.49              32.0             1

In [25]:
# Partial weeks at each end: 1 March is a Sunday and 30 June a Tuesday
gold_weekly_ops.groupby("week_start")["orders"].sum()

week_start
2026-02-23    122
2026-03-02    742
2026-03-09    814
2026-03-16    760
2026-03-23    811
2026-03-30    825
2026-04-06    829
2026-04-13    766
2026-04-20    777
2026-04-27    739
2026-05-04    802
2026-05-11    756
2026-05-18    801
2026-05-25    808
2026-06-01    778
2026-06-08    717
2026-06-15    775
2026-06-22    788
2026-06-29    214

The first week (starting 23 Feb) holds only Sunday 1 March, and the last (starting 29 June) only two days. They are **partial weeks** and should not be compared with full weeks.

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Each check compares the gold table with a number worked out a **different way** from the raw source tables.

In [26]:
# Check 1: order count. Gold total vs. orders without a 'cancelled' event, counted from raw events
cancelled_ids = events.loc[events["status"] == "cancelled", "order_id"].unique()
source_orders = (~orders["order_id"].isin(cancelled_ids)).sum()

print("gold orders:  ", gold_weekly_ops["orders"].sum())
print("source orders:", source_orders, f"(= {len(orders):,} - {len(cancelled_ids)} cancelled)")

gold orders:   13624
source orders: 13624 (= 14,494 - 870 cancelled)


In [27]:
# Check 2: net revenue. Gold total vs. the sum of every (de-duplicated) payment.
# Cancelled orders were fully refunded, so including them adds exactly zero.
print("gold net revenue:  ", round(gold_weekly_ops["net_revenue"].sum(), 2))
print("all payments, net: ", round(payments.drop_duplicates()["amount"].sum(), 2))
print("with the duplicates left in it would have been:", round(payments["amount"].sum(), 2))

gold net revenue:   507780.31
all payments, net:  507780.31
with the duplicates left in it would have been: 508926.07


In [28]:
# Check 3: revenue a third way, from what was ordered: food + delivery fee - refunds, non-cancelled orders only
kept = ~orders["order_id"].isin(cancelled_ids)
food_and_fee = (food + orders.set_index("order_id")["delivery_fee"])[kept.values].sum()
refunds_kept = pay.loc[(pay["payment_type"] == "refund") & ~pay["order_id"].isin(cancelled_ids), "amount"].sum()
print("food + fee - refunds:", round(food_and_fee + refunds_kept, 2))

food + fee - refunds: 507780.31


In [29]:
# Check 4: every cancelled order nets to zero (so leaving them out loses no revenue)
cancelled_net = silver_orders.loc[silver_orders["is_cancelled"], "net_revenue"].round(2)
print("cancelled orders:", len(cancelled_net), "| net revenue not zero:", (cancelled_net != 0).sum())

cancelled orders: 870 | net revenue not zero: 0


In [30]:
# Check 5: recalculate one gold cell by hand from the raw tables
# London Indian restaurants in the week starting 4 May 2026 (no moved restaurants involved)
cell = gold_weekly_ops[(gold_weekly_ops["week_start"] == "2026-05-04") &
                       (gold_weekly_ops["cuisine"] == "Indian") & (gold_weekly_ops["city"] == "London")]
display(cell)

rest_ids = silver_restaurants.loc[(silver_restaurants["cuisine"] == "Indian") & (silver_restaurants["city"] == "London"), "restaurant_id"]
uk_time = pd.to_datetime(orders["order_ts"], utc=True).dt.tz_convert("Europe/London")
in_week = (uk_time >= pd.Timestamp("2026-05-04", tz="Europe/London")) & (uk_time < pd.Timestamp("2026-05-11", tz="Europe/London"))
hand = orders[in_week & orders["restaurant_id"].isin(rest_ids) & ~orders["order_id"].isin(cancelled_ids)]

print("by hand: orders", len(hand), "| net revenue", round(pay[pay["order_id"].isin(hand["order_id"])]["amount"].sum(), 2))

by hand: orders 33 | net revenue 1162.88
    week_start cuisine    city  orders  net_revenue  avg_delivery_min  timed_orders
295 2026-05-04  Indian  London      33      1162.88              37.9            33


In [31]:
# Check 6: spot check one order end to end, against every raw table
oid = "O002436"   # one of the orders with a negative delivery time
display(silver_orders[silver_orders["order_id"] == oid])
display(events[events["order_id"] == oid].sort_values("event_ts"))
display(payments[payments["order_id"] == oid])
display(lines[lines["order_id"] == oid])

     order_id customer_id restaurant_id  cuisine        city                  order_ts               order_ts_uk week_start  is_cancelled  is_delivered  delivery_min  delivery_time_valid  delivery_fee  charge  refund  net_revenue
2435  O002436      C03069          R049  Italian  Birmingham 2026-03-21 19:10:00+00:00 2026-03-21 19:10:00+00:00 2026-03-16         False          True         -55.0                False          2.99   46.99     0.0        46.99
      event_id order_id     status              event_ts
30278  E009457  O002436  delivered  2026-03-21T18:15:00Z
21650  E009526  O002436     placed  2026-03-21T19:10:00Z
5252   E009530  O002436   accepted  2026-03-21T19:13:58Z
40498  E009543  O002436  picked_up  2026-03-21T19:25:20Z
     payment_id order_id payment_type  amount              paid_at
3615    P002753  O002436       charge   46.99  2026-03-21 19:11:29
     order_id  line_no     item_name  quantity  unit_price  line_value
6062  O002436        1  Cheeseburger         3    

In [32]:
# Check 7: keys and nulls in the clean and gold tables
print("silver_orders: order_id unique", silver_orders["order_id"].is_unique,
      "| missing city", silver_orders["city"].isna().sum(), "| missing net_revenue", silver_orders["net_revenue"].isna().sum())
print("gold: one row per week/cuisine/city", not gold_weekly_ops.duplicated(["week_start", "cuisine", "city"]).any())
print("gold rows with no delivery time:", gold_weekly_ops["avg_delivery_min"].isna().sum())

silver_orders: order_id unique True | missing city 0 | missing net_revenue 0
gold: one row per week/cuisine/city True
gold rows with no delivery time: 1


In [33]:
# Check 8: is the average delivery time sensitive to my 0-180 minute rule?
delivered = silver_orders[silver_orders["is_delivered"]]
print("mean, all delivered orders:     ", round(delivered["delivery_min"].mean(), 1))
print("mean, valid times only:         ", round(delivered.loc[delivered["delivery_time_valid"], "delivery_min"].mean(), 1))
print("median, all delivered orders:   ", round(delivered["delivery_min"].median(), 1))
print("smallest number of timed orders in a gold row:", gold_weekly_ops["timed_orders"].min())

mean, all delivered orders:      36.4
mean, valid times only:          35.9
median, all delivered orders:    35.4
smallest number of timed orders in a gold row: 0


**What the checks showed**

1. **Orders:** gold has 13,624 orders, the same as the 14,494 source orders minus the 870 cancelled.
2. **Revenue:** gold net revenue (£507,780.31) matches the sum of every payment exactly. With the 30 duplicate payments left in, revenue would have been overstated by £1,145.76.
3. **Revenue a third way:** food + delivery fee − refunds on non-cancelled orders gives the same total, so the payments and the order lines agree.
4. **Cancelled orders:** all 870 net to exactly £0. Leaving them out changes the order count but not the revenue.
5. **One cell by hand:** London / Indian / week of 4 May, rebuilt straight from the raw tables, gives the same order count and revenue.
6. **One order end to end:** O002436 shows the delivered event logged before the order was placed. That is why its delivery time is excluded.
7. **Keys:** every order appears once in silver, every week/cuisine/city appears once in gold, and no order is missing a city or a revenue figure.
8. **Sensitivity:** the 28 excluded times would pull the mean up from 35.9 to 36.4 minutes if left in. The median (35.4) is close to the cleaned mean, so the 0-180 minute rule changes the overall figure only a little. It can matter more for small groups.

**What the checks caught:**
- **Duplicate payments:** they would have overstated revenue.
- **Moved restaurants:** a plain join on `restaurant_id` would have duplicated orders for R007, R021 and R040 and put them in both cities.
- **UTC weeks:** using UTC would have moved 67 orders into the wrong week.

**Interpretation**

The gold table can be trusted for **order counts and net revenue**: both reconcile exactly to the source data. The **average delivery time** is reliable at the level of a city or a cuisine. In single week/cuisine/city cells some averages rest on only a handful of orders, which is why `timed_orders` is shown. One cell has no believable delivery time at all, so its average is blank. The first and last weeks are partial. Net revenue is money taken from customers after refunds. It is **not** Plateful's own income, because the data has no commission figures.

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

This attempt was written by Claude (an AI assistant) from the prompt: *"from this i want you to answer question 1 to 6, write out what you are doing, explain, and code"*. In the real test, paste your own prompts here.

Where a quick, unchecked approach would have gone wrong, and the check that caught it:
- **Joining restaurants on `restaurant_id` alone** duplicates orders for the three restaurants that moved. Caught by the row count after the join (step 4) and fixed with an as-of join.
- **Summing `payments` as delivered** overstates revenue because of 30 re-sent rows. Caught by `payment_id` not being unique (section 1).
- **Taking weeks from the UTC timestamp** puts 67 orders in the wrong UK week. Caught by comparing the two week columns.
- **Averaging every delivery time** includes negative and 20-hour deliveries. Caught by binning `delivery_min` before averaging.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

- **The 215 orders with no end status** (picked up, never delivered or cancelled). I counted them as orders because they were paid and not refunded, but I could not confirm the food arrived. I would ask Operations how the app records a failed delivery.
- **The 0-180 minute rule** for a believable delivery time is my judgement, not a business rule. I would agree a threshold with Operations.
- **Which duplicate `delivered` event is real.** I used the first. Some pairs are up to 3 minutes apart, so a few delivery times could be off by that much.
- **Net revenue is customer money, not Plateful's income.** Commission rates are not in the data.
- **The week definition** (Monday start, UK time) and assigning refunds to the order's week are my choices. The report should state them.
- **Restaurant moves are dated by day only.** An order on the day of a move takes the new city. I did not check the exact time of the move.
- **Not done (step 5, stretch):** cancelled orders and a refund rate in the gold table, and automated checks for the clean tables. A natural next step is to turn the section 4 checks into functions that run every time the pipeline is built.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.